## 4-Class Classification (Very Low / Low / Medium / High)

## 실험 설계

### 전처리
- **Buffa 컬럼 제거**: 모든 값이 0 → LDA 공분산 행렬 계산 불가
- **StandardScaler 적용**: 변수 스케일 차이 큼 → 표준화 필수
- **Target 범주화**: 사분위수 기반 4 클래스
  - Very Low: 0 ~ Q1 (1.08)
  - Low: Q1 ~ Q2 (2.15)
  - Medium: Q2 ~ Q3 (7.50)
  - High: Q3 ~ Max (27.07)

### Train/Test Split
- Train: 80% (378개)
- Test: 20% (95개)
- Stratified split (클래스 비율 유지)

### 실험 Matrix (6가지)

| 실험 | 온도 변수 | Livestock 변환 |
|------|-----------|----------------|
| Exp1 | 3개 모두 | 없음 |
| Exp2 | avg만 | 없음 |
| Exp3 | 3개 모두 | log(x+1) |
| Exp4 | avg만 | log(x+1) |
| Exp5 | PCA(1개) | 없음 |
| Exp6 | PCA(1개) | log(x+1) |

In [1]:
# 필요한 라이브러리 import
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, 
    balanced_accuracy_score, 
    f1_score, 
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)
import warnings
warnings.filterwarnings('ignore')

# 시각화 설정
plt.rcParams['figure.figsize'] = (12, 6)
sns.set_style('whitegrid')

print("라이브러리 로드 완료!")

라이브러리 로드 완료!


## 1. 데이터 로드 및 기본 전처리

In [2]:
# 데이터 로드
data = pd.read_csv('../../data/processed/cleaned_disease_dataset_2015landuse.csv')

print(f"원본 데이터 Shape: {data.shape}")
print(f"\n컬럼 목록:")
print(data.columns.tolist())
print(f"\nInfection Rate 통계:")
print(data['infection_rate'].describe())

원본 데이터 Shape: (473, 17)

컬럼 목록:
['year', 'country', 'admin', 'infection_rate', 'max_temperature', 'min_temperature', 'avg_temperature', 'Buffa', 'Cattl', 'Chick', 'Ducks', 'Goats', 'Horse', 'Sheep', 'Swine', 'rate_landuse', 'missing_ratio']

Infection Rate 통계:
count    473.000000
mean       5.420915
std        6.599666
min        0.000000
25%        1.075992
50%        2.145923
75%        7.500000
max       27.073403
Name: infection_rate, dtype: float64


### 1.1 Target 범주화 (4 클래스)

In [3]:
# 사분위수 계산
q1 = data['infection_rate'].quantile(0.25)
q2 = data['infection_rate'].quantile(0.50)
q3 = data['infection_rate'].quantile(0.75)
max_val = data['infection_rate'].max()

print(f"사분위수 값:")
print(f"Q1 (25%): {q1:.4f}")
print(f"Q2 (50%): {q2:.4f}")
print(f"Q3 (75%): {q3:.4f}")
print(f"Max: {max_val:.4f}")

# Target 범주화
bins = [0, q1, q2, q3, max_val + 0.01]
labels = ['Very Low', 'Low', 'Medium', 'High']

data['infection_class'] = pd.cut(data['infection_rate'], 
                                  bins=bins, 
                                  labels=labels, 
                                  include_lowest=True)

print(f"\n클래스별 분포:")
print(data['infection_class'].value_counts().sort_index())
print(f"\n클래스별 비율:")
print(data['infection_class'].value_counts(normalize=True).sort_index())

사분위수 값:
Q1 (25%): 1.0760
Q2 (50%): 2.1459
Q3 (75%): 7.5000
Max: 27.0734

클래스별 분포:
infection_class
Very Low    119
Low         118
Medium      118
High        118
Name: count, dtype: int64

클래스별 비율:
infection_class
Very Low    0.251586
Low         0.249471
Medium      0.249471
High        0.249471
Name: proportion, dtype: float64


### 1.2 Feature/Target 분리 및 Buffa 제거

In [4]:
# Key 컬럼 제거
key_cols = ['year', 'country', 'admin']
target_cols = ['infection_rate', 'infection_class']

# Feature 컬럼 추출
feature_cols = [col for col in data.columns if col not in key_cols + target_cols]

print(f"원본 Feature 컬럼 ({len(feature_cols)}개):")
print(feature_cols)

# Buffa, missing_ratio 제거 (모든 값이 0인 컬럼)
remove_cols = ['Buffa', 'missing_ratio']
for col in remove_cols:
    if col in feature_cols:
        feature_cols.remove(col)

print(f"\n불필요한 컬럼 제거 후 Feature 컬럼 ({len(feature_cols)}개):")
print(feature_cols)

# X, y 생성
X = data[feature_cols].copy()
y = data['infection_class'].copy()

print(f"\nX shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\ny value counts:")
print(y.value_counts().sort_index())

원본 Feature 컬럼 (13개):
['max_temperature', 'min_temperature', 'avg_temperature', 'Buffa', 'Cattl', 'Chick', 'Ducks', 'Goats', 'Horse', 'Sheep', 'Swine', 'rate_landuse', 'missing_ratio']

불필요한 컬럼 제거 후 Feature 컬럼 (11개):
['max_temperature', 'min_temperature', 'avg_temperature', 'Cattl', 'Chick', 'Ducks', 'Goats', 'Horse', 'Sheep', 'Swine', 'rate_landuse']

X shape: (473, 11)
y shape: (473,)

y value counts:
infection_class
Very Low    119
Low         118
Medium      118
High        118
Name: count, dtype: int64


### 1.3 Train/Test Split (Stratified)

In [5]:
# Stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print(f"Train set: {X_train.shape[0]}개")
print(f"Test set: {X_test.shape[0]}개")
print(f"\nTrain set 클래스 분포:")
print(y_train.value_counts().sort_index())
print(f"\nTest set 클래스 분포:")
print(y_test.value_counts().sort_index())

Train set: 378개
Test set: 95개

Train set 클래스 분포:
infection_class
Very Low    95
Low         94
Medium      94
High        95
Name: count, dtype: int64

Test set 클래스 분포:
infection_class
Very Low    24
Low         24
Medium      24
High        23
Name: count, dtype: int64


## 2. 전처리 함수 정의

In [35]:
def prepare_features(X_train, X_test, config):
    """
    실험 설정에 따라 features 준비
    
    Parameters:
    -----------
    X_train, X_test : DataFrame
    config : dict
        - 'temp_vars': '3개' or 'avg' or 'pca'
        - 'livestock_log': True or False
    
    Returns:
    --------
    X_train_scaled, X_test_scaled : array
    """
    X_train_prep = X_train.copy()
    X_test_prep = X_test.copy()
    
    # 온도 변수
    temp_cols = ['max_temperature', 'min_temperature', 'avg_temperature']
    
    # Livestock 변수 (온도 변수, rate_landuse 제외한 나머지)
    livestock_cols = [col for col in X_train.columns if col not in temp_cols + ['rate_landuse']]
    
    # 1. Livestock log 변환
    if config['livestock_log']:
        for col in livestock_cols:
            X_train_prep[col] = np.log1p(X_train_prep[col])
            X_test_prep[col] = np.log1p(X_test_prep[col])
    
    # 2. 온도 변수 처리
    if config['temp_vars'] == 'avg':
        # avg_temperature만 사용
        X_train_prep = X_train_prep.drop(columns=['max_temperature', 'min_temperature'])
        X_test_prep = X_test_prep.drop(columns=['max_temperature', 'min_temperature'])

    elif config['temp_vars'] == 'min':  # ⭐ 새로 추가!
        # min_temperature만 사용
        X_train_prep = X_train_prep.drop(columns=['max_temperature', 'avg_temperature'])
        X_test_prep = X_test_prep.drop(columns=['max_temperature', 'avg_temperature'])
        
        
    elif config['temp_vars'] == 'pca':
        # 온도 3개 변수를 PCA로 1개로 축소
        pca = PCA(n_components=1)
        
        # Train set PCA
        temp_train_pca = pca.fit_transform(X_train_prep[temp_cols])
        temp_test_pca = pca.transform(X_test_prep[temp_cols])
        
        # 기존 온도 변수 제거 후 PCA 컴포넌트 추가
        X_train_prep = X_train_prep.drop(columns=temp_cols)
        X_test_prep = X_test_prep.drop(columns=temp_cols)
        
        X_train_prep['temp_pca'] = temp_train_pca
        X_test_prep['temp_pca'] = temp_test_pca
    
    # 3개 모두 사용하는 경우는 그대로 유지
    
    # 3. StandardScaler 적용
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_prep)
    X_test_scaled = scaler.transform(X_test_prep)
    
    return X_train_scaled, X_test_scaled, X_train_prep.columns.tolist()

In [31]:
def evaluate_lda(X_train_scaled, X_test_scaled, y_train, y_test, cv=5):
    """
    LDA 모델 학습 및 평가
    
    Returns:
    --------
    results : dict
        - cv_scores: CV 점수 (list)
        - cv_mean: CV 평균
        - cv_std: CV 표준편차
        - test_accuracy: Test accuracy
        - test_balanced_acc: Test balanced accuracy
        - test_f1: Test F1-score (macro)
        - confusion_matrix: Confusion matrix
        - classification_report: 상세 리포트
        - model: 학습된 모델
    """
    # LDA 모델
    lda = LinearDiscriminantAnalysis(solver='svd')
    
    # 5-Fold Cross Validation
    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=42)
    cv_scores = cross_val_score(lda, X_train_scaled, y_train, cv=skf, scoring='accuracy')
    
    # Train set 전체로 최종 모델 학습
    lda.fit(X_train_scaled, y_train)
    
    # Test set 예측
    y_pred = lda.predict(X_test_scaled)
    
    # 평가 지표 계산
    results = {
        'cv_scores': cv_scores,
        'cv_mean': cv_scores.mean(),
        'cv_std': cv_scores.std(),
        'test_accuracy': accuracy_score(y_test, y_pred),
        'test_balanced_acc': balanced_accuracy_score(y_test, y_pred),
        'test_f1': f1_score(y_test, y_pred, average='macro'),
        'confusion_matrix': confusion_matrix(y_test, y_pred, labels=['Very Low', 'Low', 'Medium', 'High']),
        'classification_report': classification_report(y_test, y_pred, target_names=['Very Low', 'Low', 'Medium', 'High']),
        'model': lda
    }
    
    return results

## 3. 실험 1: 온도 3개 모두 + Livestock 원본

In [8]:
print("="*70)
print(" "*20 + "실험 1: 온도 3개 모두 + Livestock 원본")
print("="*70)

config_exp1 = {
    'temp_vars': '3개',
    'livestock_log': False
}

X_train_exp1, X_test_exp1, features_exp1 = prepare_features(X_train, X_test, config_exp1)
print(f"\nFeature 개수: {len(features_exp1)}개")
print(f"Features: {features_exp1}")

results_exp1 = evaluate_lda(X_train_exp1, X_test_exp1, y_train, y_test)

print(f"\n[Cross-Validation 결과]")
print(f"CV Scores: {results_exp1['cv_scores']}")
print(f"CV Mean: {results_exp1['cv_mean']:.4f}")
print(f"CV Std: {results_exp1['cv_std']:.4f}")

print(f"\n[Test Set 결과]")
print(f"Accuracy: {results_exp1['test_accuracy']:.4f}")
print(f"Balanced Accuracy: {results_exp1['test_balanced_acc']:.4f}")
print(f"F1-Score (macro): {results_exp1['test_f1']:.4f}")

print(f"\n[Classification Report]")
print(results_exp1['classification_report'])

                    실험 1: 온도 3개 모두 + Livestock 원본

Feature 개수: 11개
Features: ['max_temperature', 'min_temperature', 'avg_temperature', 'Cattl', 'Chick', 'Ducks', 'Goats', 'Horse', 'Sheep', 'Swine', 'rate_landuse']

[Cross-Validation 결과]
CV Scores: [0.46052632 0.51315789 0.51315789 0.52       0.52      ]
CV Mean: 0.5054
CV Std: 0.0226

[Test Set 결과]
Accuracy: 0.4947
Balanced Accuracy: 0.4955
F1-Score (macro): 0.5151

[Classification Report]
              precision    recall  f1-score   support

    Very Low       0.93      0.57      0.70        23
         Low       0.23      0.29      0.25        24
      Medium       0.50      0.54      0.52        24
        High       0.58      0.58      0.58        24

    accuracy                           0.49        95
   macro avg       0.56      0.50      0.52        95
weighted avg       0.56      0.49      0.51        95



## 4. 실험 2: avg_temperature만 + Livestock 원본

In [32]:
print("="*70)
print(" "*20 + "실험 2: avg_temperature만 + Livestock 원본")
print("="*70)

config_exp2 = {
    'temp_vars': 'avg',
    'livestock_log': False
}

X_train_exp2, X_test_exp2, features_exp2 = prepare_features(X_train, X_test, config_exp2)
print(f"\nFeature 개수: {len(features_exp2)}개")
print(f"Features: {features_exp2}")

results_exp2 = evaluate_lda(X_train_exp2, X_test_exp2, y_train, y_test)

print(f"\n[Cross-Validation 결과]")
print(f"CV Scores: {results_exp2['cv_scores']}")
print(f"CV Mean: {results_exp2['cv_mean']:.4f}")
print(f"CV Std: {results_exp2['cv_std']:.4f}")

print(f"\n[Test Set 결과]")
print(f"Accuracy: {results_exp2['test_accuracy']:.4f}")
print(f"Balanced Accuracy: {results_exp2['test_balanced_acc']:.4f}")
print(f"F1-Score (macro): {results_exp2['test_f1']:.4f}")

print(f"\n[Classification Report]")
print(results_exp2['classification_report'])

                    실험 2: avg_temperature만 + Livestock 원본

Feature 개수: 9개
Features: ['min_temperature', 'Cattl', 'Chick', 'Ducks', 'Goats', 'Horse', 'Sheep', 'Swine', 'rate_landuse']

[Cross-Validation 결과]
CV Scores: [0.35526316 0.46052632 0.57894737 0.49333333 0.53333333]
CV Mean: 0.4843
CV Std: 0.0757

[Test Set 결과]
Accuracy: 0.4737
Balanced Accuracy: 0.4746
F1-Score (macro): 0.4987

[Classification Report]
              precision    recall  f1-score   support

    Very Low       1.00      0.57      0.72        23
         Low       0.29      0.50      0.36        24
      Medium       0.40      0.33      0.36        24
        High       0.60      0.50      0.55        24

    accuracy                           0.47        95
   macro avg       0.57      0.47      0.50        95
weighted avg       0.57      0.47      0.50        95



## 5. 실험 3: 온도 3개 모두 + Livestock log 변환

In [10]:
print("="*70)
print(" "*20 + "실험 3: 온도 3개 모두 + Livestock log 변환")
print("="*70)

config_exp3 = {
    'temp_vars': '3개',
    'livestock_log': True
}

X_train_exp3, X_test_exp3, features_exp3 = prepare_features(X_train, X_test, config_exp3)
print(f"\nFeature 개수: {len(features_exp3)}개")
print(f"Features: {features_exp3}")

results_exp3 = evaluate_lda(X_train_exp3, X_test_exp3, y_train, y_test)

print(f"\n[Cross-Validation 결과]")
print(f"CV Scores: {results_exp3['cv_scores']}")
print(f"CV Mean: {results_exp3['cv_mean']:.4f}")
print(f"CV Std: {results_exp3['cv_std']:.4f}")

print(f"\n[Test Set 결과]")
print(f"Accuracy: {results_exp3['test_accuracy']:.4f}")
print(f"Balanced Accuracy: {results_exp3['test_balanced_acc']:.4f}")
print(f"F1-Score (macro): {results_exp3['test_f1']:.4f}")

print(f"\n[Classification Report]")
print(results_exp3['classification_report'])

                    실험 3: 온도 3개 모두 + Livestock log 변환

Feature 개수: 11개
Features: ['max_temperature', 'min_temperature', 'avg_temperature', 'Cattl', 'Chick', 'Ducks', 'Goats', 'Horse', 'Sheep', 'Swine', 'rate_landuse']

[Cross-Validation 결과]
CV Scores: [0.48684211 0.61842105 0.63157895 0.56       0.50666667]
CV Mean: 0.5607
CV Std: 0.0578

[Test Set 결과]
Accuracy: 0.5579
Balanced Accuracy: 0.5611
F1-Score (macro): 0.5575

[Classification Report]
              precision    recall  f1-score   support

    Very Low       0.95      0.87      0.91        23
         Low       0.35      0.25      0.29        24
      Medium       0.48      0.54      0.51        24
        High       0.47      0.58      0.52        24

    accuracy                           0.56        95
   macro avg       0.56      0.56      0.56        95
weighted avg       0.56      0.56      0.55        95



## 6. 실험 4: avg_temperature만 + Livestock log 변환

In [33]:
print("="*70)
print(" "*20 + "실험 4: avg_temperature만 + Livestock log 변환")
print("="*70)

config_exp4 = {
    'temp_vars': 'avg',
    'livestock_log': True
}

X_train_exp4, X_test_exp4, features_exp4 = prepare_features(X_train, X_test, config_exp4)
print(f"\nFeature 개수: {len(features_exp4)}개")
print(f"Features: {features_exp4}")

results_exp4 = evaluate_lda(X_train_exp4, X_test_exp4, y_train, y_test)

print(f"\n[Cross-Validation 결과]")
print(f"CV Scores: {results_exp4['cv_scores']}")
print(f"CV Mean: {results_exp4['cv_mean']:.4f}")
print(f"CV Std: {results_exp4['cv_std']:.4f}")

print(f"\n[Test Set 결과]")
print(f"Accuracy: {results_exp4['test_accuracy']:.4f}")
print(f"Balanced Accuracy: {results_exp4['test_balanced_acc']:.4f}")
print(f"F1-Score (macro): {results_exp4['test_f1']:.4f}")

print(f"\n[Classification Report]")
print(results_exp4['classification_report'])

                    실험 4: avg_temperature만 + Livestock log 변환

Feature 개수: 9개
Features: ['min_temperature', 'Cattl', 'Chick', 'Ducks', 'Goats', 'Horse', 'Sheep', 'Swine', 'rate_landuse']

[Cross-Validation 결과]
CV Scores: [0.46052632 0.63157895 0.61842105 0.62666667 0.54666667]
CV Mean: 0.5768
CV Std: 0.0658

[Test Set 결과]
Accuracy: 0.5684
Balanced Accuracy: 0.5716
F1-Score (macro): 0.5581

[Classification Report]
              precision    recall  f1-score   support

    Very Low       0.91      0.87      0.89        23
         Low       0.33      0.21      0.26        24
      Medium       0.52      0.50      0.51        24
        High       0.49      0.71      0.58        24

    accuracy                           0.57        95
   macro avg       0.56      0.57      0.56        95
weighted avg       0.56      0.57      0.55        95



## 7. 실험 5: 온도 PCA(1개) + Livestock 원본

In [12]:
print("="*70)
print(" "*20 + "실험 5: 온도 PCA(1개) + Livestock 원본")
print("="*70)

config_exp5 = {
    'temp_vars': 'pca',
    'livestock_log': False
}

X_train_exp5, X_test_exp5, features_exp5 = prepare_features(X_train, X_test, config_exp5)
print(f"\nFeature 개수: {len(features_exp5)}개")
print(f"Features: {features_exp5}")

results_exp5 = evaluate_lda(X_train_exp5, X_test_exp5, y_train, y_test)

print(f"\n[Cross-Validation 결과]")
print(f"CV Scores: {results_exp5['cv_scores']}")
print(f"CV Mean: {results_exp5['cv_mean']:.4f}")
print(f"CV Std: {results_exp5['cv_std']:.4f}")

print(f"\n[Test Set 결과]")
print(f"Accuracy: {results_exp5['test_accuracy']:.4f}")
print(f"Balanced Accuracy: {results_exp5['test_balanced_acc']:.4f}")
print(f"F1-Score (macro): {results_exp5['test_f1']:.4f}")

print(f"\n[Classification Report]")
print(results_exp5['classification_report'])

                    실험 5: 온도 PCA(1개) + Livestock 원본

Feature 개수: 9개
Features: ['Cattl', 'Chick', 'Ducks', 'Goats', 'Horse', 'Sheep', 'Swine', 'rate_landuse', 'temp_pca']

[Cross-Validation 결과]
CV Scores: [0.38157895 0.47368421 0.55263158 0.49333333 0.50666667]
CV Mean: 0.4816
CV Std: 0.0564

[Test Set 결과]
Accuracy: 0.4526
Balanced Accuracy: 0.4529
F1-Score (macro): 0.4713

[Classification Report]
              precision    recall  f1-score   support

    Very Low       0.92      0.48      0.63        23
         Low       0.29      0.42      0.34        24
      Medium       0.39      0.46      0.42        24
        High       0.52      0.46      0.49        24

    accuracy                           0.45        95
   macro avg       0.53      0.45      0.47        95
weighted avg       0.53      0.45      0.47        95



## 8. 실험 6: 온도 PCA(1개) + Livestock log 변환

In [13]:
print("="*70)
print(" "*20 + "실험 6: 온도 PCA(1개) + Livestock log 변환")
print("="*70)

config_exp6 = {
    'temp_vars': 'pca',
    'livestock_log': True
}

X_train_exp6, X_test_exp6, features_exp6 = prepare_features(X_train, X_test, config_exp6)
print(f"\nFeature 개수: {len(features_exp6)}개")
print(f"Features: {features_exp6}")

results_exp6 = evaluate_lda(X_train_exp6, X_test_exp6, y_train, y_test)

print(f"\n[Cross-Validation 결과]")
print(f"CV Scores: {results_exp6['cv_scores']}")
print(f"CV Mean: {results_exp6['cv_mean']:.4f}")
print(f"CV Std: {results_exp6['cv_std']:.4f}")

print(f"\n[Test Set 결과]")
print(f"Accuracy: {results_exp6['test_accuracy']:.4f}")
print(f"Balanced Accuracy: {results_exp6['test_balanced_acc']:.4f}")
print(f"F1-Score (macro): {results_exp6['test_f1']:.4f}")

print(f"\n[Classification Report]")
print(results_exp6['classification_report'])

                    실험 6: 온도 PCA(1개) + Livestock log 변환

Feature 개수: 9개
Features: ['Cattl', 'Chick', 'Ducks', 'Goats', 'Horse', 'Sheep', 'Swine', 'rate_landuse', 'temp_pca']

[Cross-Validation 결과]
CV Scores: [0.48684211 0.55263158 0.60526316 0.62666667 0.53333333]
CV Mean: 0.5609
CV Std: 0.0502

[Test Set 결과]
Accuracy: 0.5158
Balanced Accuracy: 0.5190
F1-Score (macro): 0.5007

[Classification Report]
              precision    recall  f1-score   support

    Very Low       0.73      0.83      0.78        23
         Low       0.33      0.21      0.26        24
      Medium       0.48      0.42      0.44        24
        High       0.45      0.62      0.53        24

    accuracy                           0.52        95
   macro avg       0.50      0.52      0.50        95
weighted avg       0.50      0.52      0.50        95



## 실험 7 : min_temperature만 + Livestock 원본

In [36]:
print("="*70)
print(" "*15 + "실험 7: min_temperature만 + Livestock 원본")
print("="*70)

config_exp7 = {
    'temp_vars': 'min',
    'livestock_log': False
}

X_train_exp7, X_test_exp7, features_exp7 = prepare_features(X_train, X_test, config_exp7)
print(f"\nFeature 개수: {len(features_exp7)}개")
print(f"Features: {features_exp7}")

results_exp7 = evaluate_lda(X_train_exp7, X_test_exp7, y_train, y_test)

print(f"\n[Cross-Validation 결과]")
print(f"CV Scores: {results_exp7['cv_scores']}")
print(f"CV Mean: {results_exp7['cv_mean']:.4f}")
print(f"CV Std: {results_exp7['cv_std']:.4f}")

print(f"\n[Test Set 결과]")
print(f"Accuracy: {results_exp7['test_accuracy']:.4f}")
print(f"Balanced Accuracy: {results_exp7['test_balanced_acc']:.4f}")
print(f"F1-Score (macro): {results_exp7['test_f1']:.4f}")

print(f"\n[Classification Report]")
print(results_exp7['classification_report'])

               실험 7: min_temperature만 + Livestock 원본

Feature 개수: 9개
Features: ['min_temperature', 'Cattl', 'Chick', 'Ducks', 'Goats', 'Horse', 'Sheep', 'Swine', 'rate_landuse']

[Cross-Validation 결과]
CV Scores: [0.35526316 0.46052632 0.57894737 0.49333333 0.53333333]
CV Mean: 0.4843
CV Std: 0.0757

[Test Set 결과]
Accuracy: 0.4737
Balanced Accuracy: 0.4746
F1-Score (macro): 0.4987

[Classification Report]
              precision    recall  f1-score   support

    Very Low       1.00      0.57      0.72        23
         Low       0.29      0.50      0.36        24
      Medium       0.40      0.33      0.36        24
        High       0.60      0.50      0.55        24

    accuracy                           0.47        95
   macro avg       0.57      0.47      0.50        95
weighted avg       0.57      0.47      0.50        95



## 실험 8 : min_temperature만 + Livestock log 변환

In [37]:
print("="*70)
print(" "*15 + "실험 8: min_temperature만 + Livestock log 변환")
print("="*70)

config_exp8 = {
    'temp_vars': 'min',
    'livestock_log': True
}

X_train_exp8, X_test_exp8, features_exp8 = prepare_features(X_train, X_test, config_exp8)
print(f"\nFeature 개수: {len(features_exp8)}개")
print(f"Features: {features_exp8}")

results_exp8 = evaluate_lda(X_train_exp8, X_test_exp8, y_train, y_test)

print(f"\n[Cross-Validation 결과]")
print(f"CV Scores: {results_exp8['cv_scores']}")
print(f"CV Mean: {results_exp8['cv_mean']:.4f}")
print(f"CV Std: {results_exp8['cv_std']:.4f}")

print(f"\n[Test Set 결과]")
print(f"Accuracy: {results_exp8['test_accuracy']:.4f}")
print(f"Balanced Accuracy: {results_exp8['test_balanced_acc']:.4f}")
print(f"F1-Score (macro): {results_exp8['test_f1']:.4f}")

print(f"\n[Classification Report]")
print(results_exp8['classification_report'])

               실험 8: min_temperature만 + Livestock log 변환

Feature 개수: 9개
Features: ['min_temperature', 'Cattl', 'Chick', 'Ducks', 'Goats', 'Horse', 'Sheep', 'Swine', 'rate_landuse']

[Cross-Validation 결과]
CV Scores: [0.46052632 0.63157895 0.61842105 0.62666667 0.54666667]
CV Mean: 0.5768
CV Std: 0.0658

[Test Set 결과]
Accuracy: 0.5684
Balanced Accuracy: 0.5716
F1-Score (macro): 0.5581

[Classification Report]
              precision    recall  f1-score   support

    Very Low       0.91      0.87      0.89        23
         Low       0.33      0.21      0.26        24
      Medium       0.52      0.50      0.51        24
        High       0.49      0.71      0.58        24

    accuracy                           0.57        95
   macro avg       0.56      0.57      0.56        95
weighted avg       0.56      0.57      0.55        95



## 15. LDA 결과 해석 (1): 판별 함수 계수 분석

LDA의 coefficients를 분석하여 각 변수가 클래스 분류에 어떻게 기여하는지 확인합니다.

In [14]:
# 최고 성능 모델(실험 3)의 계수 분석
best_model = results_exp3['model']

print("="*80)
print(" "*25 + "LDA 판별 함수 계수 분석")
print("="*80)

# Feature names (실험 3 기준)
feature_names = features_exp3

# Coefficients (4 클래스 × 11 features)
coefficients = best_model.coef_
print(f"\nCoefficients shape: {coefficients.shape}")
print(f"(4 classes × {len(feature_names)} features)\n")

# 각 클래스별 계수를 DataFrame으로 정리
coef_df = pd.DataFrame(
    coefficients.T,
    index=feature_names,
    columns=['Very Low', 'Low', 'Medium', 'High']
)

print("\n[클래스별 판별 계수]")
print(coef_df.round(4))

# 각 변수의 중요도 (계수의 절댓값 평균)
coef_df['Importance'] = coef_df.abs().mean(axis=1)
coef_df_sorted = coef_df.sort_values('Importance', ascending=False)

print("\n" + "="*80)
print(" "*25 + "변수 중요도 순위")
print("="*80)
print(coef_df_sorted[['Importance']].round(4))

                         LDA 판별 함수 계수 분석

Coefficients shape: (4, 11)
(4 classes × 11 features)


[클래스별 판별 계수]
                 Very Low     Low  Medium    High
max_temperature    0.7243  0.6620 -0.7921 -0.5956
min_temperature   -2.2058  1.1449  0.7743  0.3068
avg_temperature   -0.4666 -0.8457  0.0806  1.2237
Cattl              0.4675 -0.1207 -0.1425 -0.2070
Chick              0.1522 -0.0400  0.2366 -0.3468
Ducks              0.3710  0.1426 -0.0919 -0.4212
Goats             -0.2152 -0.1508 -0.5531  0.9117
Horse             -0.0107 -0.0260 -0.3533  0.3860
Sheep             -1.5250  0.6223  0.4830  0.4313
Swine              0.2400 -0.0905  0.0614 -0.2112
rate_landuse       0.1597 -0.1563  0.2177 -0.2205

                         변수 중요도 순위
                 Importance
min_temperature      1.1079
Sheep                0.7654
max_temperature      0.6935
avg_temperature      0.6542
Goats                0.4577
Ducks                0.2567
Cattl                0.2344
Horse                0.1940
C